# P&ID Medallion Pipeline — Concepts Walkthrough (Bronze → Silver)

This notebook illustrates, end to end, what we have built so far: a **Bronze**
(raw, immutable ingestion) → **Silver** (parse + topology reconstruction) pipeline
for P&ID interoperability exports (DEXPI/Proteus and INGR ISO-15926 PostProc),
on local Spark + Delta Lake.

It demonstrates the key concepts:

- **Bronze** stores the source XML *as-is* — content hash, format detection, project
  code and drawing revision captured, dedup on exact bytes.
- **Silver** *re-houses* the validated `pidtool`/`bppidsys` reconstruction (the
  "crown jewel") — recovering inline valves the raw graph lacks — and emits typed
  tables: components, segments, connections, equipment.
- The **oracle firewall**: the source turnover assignment is carried as *quarantined*
  lineage, never computed on.
- The **`flow_sense`** four-state directional overlay and the **`derived`** provenance
  flag on every reified connection.
- **Format parity**: DEXPI and PostProc flow through one code path into one schema.
- A real-data finding: **`seg_tag` is not unique** (the CDC anchor-collision risk).

> **Run order matters.** After any kernel restart, run the cells top-to-bottom.
> Cell 1 *must* be first — it forces the venv's Spark 3.5.1 and blocks the system
> Spark 4 at `/opt/spark`.

## 0. Environment & pinned session

Two things bite on local WSL and are handled here:

1. **Which Spark.** A system `SPARK_HOME=/opt/spark` (Spark 4) shadows the venv's
   Spark 3.5.1 and breaks Delta (`DeltaCatalog` not found). Cell 1 strips it
   *before* `pyspark` is ever imported.
2. **One catalog, one warehouse.** The Hive metastore (`metastore_db/`) holds
   *names → locations*; the warehouse (`spark-warehouse/`) holds the *data*. We
   **pin both** to fixed paths so every session sees the same tables (embedded
   Derby is single-session — don't also run a `!python -m ...` subprocess while
   this notebook's session is live).

In [1]:
# --- CELL 1 — must run FIRST (before any `import pyspark`) ---
import os, sys
os.environ.pop("SPARK_HOME", None)                       # ignore system /opt/spark (Spark 4)
os.environ["PYTHONPATH"] = os.pathsep.join(
    p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if "/opt/spark" not in p)
sys.path[:] = [p for p in sys.path if "/opt/spark" not in p]
assert "pyspark" not in sys.modules, "Restart the kernel and run THIS cell first."

from pathlib import Path

# repo_root = the ProjectData repo root (the folder containing bronze/ and silver/)
repo_root = Path.cwd()
while not (repo_root / "bronze").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
assert (repo_root / "bronze").is_dir(), f"Open this notebook inside the ProjectData repo (cwd={Path.cwd()})"

# --- the variables for this walkthrough ---
source_sample_dir = repo_root / "sample_data"                 # committed synthetic fixtures
source_dir        = repo_root / "data/exports/projectA"       # real Project A (DEXPI)
source_dir_B      = repo_root / "data/exports/projectB"       # real Project B (PostProc)
table_path        = repo_root / "_tmp" / "bronze_pid_documents"  # PATH-BASED Bronze (throwaway)
spark_warehouse   = repo_root / "spark-warehouse"             # managed-table data
metastore_db      = repo_root / "metastore_db"                # Hive/Derby catalog

print("repo_root       :", repo_root)
print("table_path      :", table_path)
print("spark_warehouse :", spark_warehouse)
print("metastore_db    :", metastore_db)

repo_root       : /home/dcamacho/dev/ProjectData
table_path      : /home/dcamacho/dev/ProjectData/_tmp/bronze_pid_documents
spark_warehouse : /home/dcamacho/dev/ProjectData/spark-warehouse
metastore_db    : /home/dcamacho/dev/ProjectData/metastore_db


In [2]:
# --- CELL 2 — build ONE pinned Delta+Hive session (venv Spark 3.5.1) ---
from bronze.spark_session import get_spark
spark = get_spark(extra_conf={
    "spark.sql.warehouse.dir": f"file:{spark_warehouse}",
    "spark.hadoop.javax.jdo.option.ConnectionURL":
        f"jdbc:derby:;databaseName={metastore_db};create=true",
})
import pyspark
from pyspark.sql import functions as F
print("pyspark :", pyspark.__file__)   # expect .../.venv/...  NOT /opt/spark
print("Spark   :", spark.version)      # expect 3.5.1
assert "/opt/spark" not in pyspark.__file__, "Still on system Spark 4 — restart kernel, run Cell 1 first."
assert spark.version.startswith("3.5"), f"Expected Spark 3.5.x, got {spark.version}"
print("OK — venv Spark 3.5.1, Delta + Hive ready.")

your 131072x1 screen size is bogus. expect trouble
26/09/02 07:29:02 WARN Utils: Your hostname, DC01NNCOL resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/02 07:29:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/dcamacho/.ivy2/cache
The jars for the packages stored in: /home/dcamacho/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0b0e3120-8e24-4d2f-ab3c-c2ee7adfa9d7;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 154ms :: artifacts dl 6ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   | 

pyspark : /home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/__init__.py
Spark   : 3.5.1
OK — venv Spark 3.5.1, Delta + Hive ready.


In [3]:
# --- CELL 3 (optional) — clean slate for a reproducible demo ---
# Safe: _tmp Bronze is throwaway; Silver tables are rebuilt from Bronze below.
import shutil
shutil.rmtree(table_path, ignore_errors=True)
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
for t in ["silver_components", "silver_segments", "silver_connections", "silver_equipment"]:
    spark.sql(f"DROP TABLE IF EXISTS silver.{t}")
    shutil.rmtree(spark_warehouse / "silver.db" / t, ignore_errors=True)
print("clean slate ready")

26/09/02 07:29:06 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/09/02 07:29:06 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/09/02 07:29:08 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/09/02 07:29:08 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore dcamacho@127.0.1.1
26/09/02 07:29:08 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


clean slate ready


## 1. Bronze — raw, immutable ingestion

Bronze lands each source file **verbatim**, one row per distinct byte-version, with:
`content` (raw bytes), a self-describing `content_hash` (`sha256:…`), the detected
`source_format` (DEXPI vs POSTPROC), the EPC `document_number` and derived
`project_code`, and the current `drawing_revision` / `drawing_revision_date`.
It **never interprets** the network model — that's Silver's job.

Here we ingest into a **path-based** Bronze table (`table_path`), which needs no
metastore at all.

In [4]:
# --- pick sources: prefer the real exports, fall back to the committed samples ---
def xmls(d): return sorted(Path(d).glob("*.xml")) if Path(d).is_dir() else []
sources = [d for d in (source_dir, source_dir_B) if xmls(d)]
if not sources:
    sources = [source_sample_dir]
for d in sources:
    print(f"{len(xmls(d)):3d} xml  in  {d}")

  4 xml  in  /home/dcamacho/dev/ProjectData/data/exports/projectA
  5 xml  in  /home/dcamacho/dev/ProjectData/data/exports/projectB


In [5]:
# --- ingest each source folder into the SAME path-based Bronze table ---
from bronze.notebook import ingest_folder
for d in sources:
    summary = ingest_folder(spark, source_dir=str(d), table_path=str(table_path))
    print(d.name, "->", summary)

26/09/02 07:29:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


projectA -> {'ingest_run_id': '679feebb-8cf6-4747-abcd-af832f54d2b1', 'files_in_batch': 4, 'rows_inserted': 4, 'rows_skipped_already_present': 0}


projectB -> {'ingest_run_id': 'd4cb1173-2af2-41b5-9cab-f4593a6a7ed1', 'files_in_batch': 5, 'rows_inserted': 5, 'rows_skipped_already_present': 0}


In [6]:
# --- inspect Bronze: both formats, lineage columns, self-describing hash ---
bronze = spark.read.format("delta").load(str(table_path))
print("Bronze rows:", bronze.count())
bronze.groupBy("source_format").count().show()
bronze.select("document_number", "drawing_revision", "drawing_revision_date",
              "project_code", "content_hash").show(6, False)

Bronze rows: 9
+-------------+-----+
|source_format|count|
+-------------+-----+
|     POSTPROC|    5|
|        DEXPI|    4|
+-------------+-----+

+-----------------------------+----------------+---------------------+------------+-----------------------------------------------------------------------+
|document_number              |drawing_revision|drawing_revision_date|project_code|content_hash                                                           |
+-----------------------------+----------------+---------------------+------------+-----------------------------------------------------------------------+
|216097C-A14-PID-0021-0005-001|E               |2026/05/08           |216097C     |sha256:03bc31a7d6ed930b1c286cacbe75693fdd6cefef29281f6e5860b21006d2bf06|
|216097C-A14-PID-0021-0004-001|E               |2026/05/08           |216097C     |sha256:cab99a0b87087921133e67189b8afbce33fd9723ce34b7c1f89e76695fba8dd5|
|216097C-A14-PID-0021-0001-001|E               |2026/05/08           |21

**Dedup on exact bytes.** Re-ingesting the same files lands *nothing* new —
Bronze versions files by `content_hash`, so identical bytes are skipped
(`rows_skipped_already_present`).

In [7]:
# re-ingest the first folder — expect rows_inserted: 0
print(ingest_folder(spark, source_dir=str(sources[0]), table_path=str(table_path)))

{'ingest_run_id': 'a34bb89b-841e-48dd-8e9e-7eab1816cbd0', 'files_in_batch': 4, 'rows_inserted': 0, 'rows_skipped_already_present': 4}


## 2. Silver — parse + topology reconstruction

Silver reads Bronze, picks the adapter from `source_format`, builds the DOM from
the raw bytes, and runs the **validated reconstruction** (vendored under
`silver/_recon/`, re-housed not re-derived). It emits four typed Delta tables and
carries the source turnover assignment as **quarantined** lineage.

We run it **in-session** (same notebook session) reading Bronze by path — so the
Silver tables land in this session's pinned catalog and are queryable by name.

In [8]:
# --- run Silver Stage A+B in-session ---
from silver.notebook import reconstruct
counts = reconstruct(spark, bronze_path=str(table_path), silver_schema="silver")
print(counts)

26/09/02 07:29:36 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver`.`silver_components` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/09/02 07:29:36 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/09/02 07:29:36 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/09/02 07:29:36 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/09/02 07:29:36 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/09/02 07:29:38 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver`.`silver_segments` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
2

{'silver_components': 3553, 'silver_segments': 1520, 'silver_connections': 2781, 'silver_equipment': 12}


In [9]:
for t in ["silver_components", "silver_segments", "silver_connections", "silver_equipment"]:
    print(f"{t:22s} {spark.table('silver.' + t).count():6d} rows")

silver_components        3553 rows
silver_segments          1520 rows
silver_connections       2781 rows
silver_equipment           12 rows


## 3. The concepts, illustrated in the data

### 3a. The crown jewel — inline valves recovered

The raw `<Connection>` records wire only each segment's two endpoints; inline valves
are missing. The reconstruction repairs the topology and re-inserts them. Here they
appear as real components flagged `is_valve` — in **both** formats.

In [10]:
spark.table("silver.silver_components") \
     .groupBy("source_format", "is_valve").count() \
     .orderBy("source_format", "is_valve").show()

+-------------+--------+-----+
|source_format|is_valve|count|
+-------------+--------+-----+
|        DEXPI|   false| 1376|
|        DEXPI|    true|  190|
|     POSTPROC|   false| 1755|
|     POSTPROC|    true|  232|
+-------------+--------+-----+



### 3b. The oracle firewall

`src_turnover` / `src_subsystem` (the source commissioning assignment) is **carried**
on the segment row — but it sits on its own columns and **nothing computes on it**.
It is the validation *answer key*, quarantined so the ~97% agreement stays honest.

In [11]:
spark.table("silver.silver_segments") \
     .select("seg_tag", "fluid", "piping_materials_class",
             "src_turnover", "src_subsystem", "project_code").show(6, False)

+--------------------+-----+----------------------+------------+-------------+------------+
|seg_tag             |fluid|piping_materials_class|src_turnover|src_subsystem|project_code|
+--------------------+-----+----------------------+------------+-------------+------------+
|2"-SV-36209-1B6AS-N |SV   |1B6AS                 |0920        |11-0920-030  |215777C     |
|2"-SV-36209-1B6AS-N |SV   |1B6AS                 |0920        |11-0920-030  |215777C     |
|2"-SV-36209-1B6AS-N |SV   |1B6AS                 |0920        |11-0920-010  |215777C     |
|2"-SV-36209-1B6AS-1 |SV   |1B6AS                 |0920        |11-0920-010  |215777C     |
|24"-SV-36209-1B6AS-1|SV   |1B6AS                 |0920        |11-0920-010  |215777C     |
|30"-SV-36209-1B6AS-1|SV   |1B6AS                 |0920        |11-0920-010  |215777C     |
+--------------------+-----+----------------------+------------+-------------+------------+
only showing top 6 rows



### 3c. `flow_sense` — the four-state directional overlay

Direction is a *separate overlay* on the undirected connection, and it has four
states — `none` and `both` are real and a boolean couldn't hold them. Both formats
produce all four.

In [12]:
spark.table("silver.silver_connections") \
     .groupBy("source_format", "flow_sense").count() \
     .orderBy("source_format", "flow_sense").show()

+-------------+----------+-----+
|source_format|flow_sense|count|
+-------------+----------+-----+
|        DEXPI|      both|    1|
|        DEXPI|   forward|  631|
|        DEXPI|      none|  121|
|        DEXPI|   reverse|  632|
|     POSTPROC|      both|    9|
|     POSTPROC|   forward|  616|
|     POSTPROC|      none|  135|
|     POSTPROC|   reverse|  636|
+-------------+----------+-----+



### 3d. `derived` — Source (stated) vs Derived (reconstructed) edges

Every reified connection carries provenance: `derived=false` where the edge was
stated in a source `<Connection>`, `derived=true` where the reconstruction inferred
it. This is what keeps the semantic layer from asserting inferred topology as fact.

In [13]:
spark.table("silver.silver_connections").groupBy("source_format", "derived").count().show()

+-------------+-------+-----+
|source_format|derived|count|
+-------------+-------+-----+
|     POSTPROC|   true|  726|
|     POSTPROC|  false|  670|
|        DEXPI|  false|  357|
|        DEXPI|   true| 1028|
+-------------+-------+-----+



### 3e. Format parity — two standards, one schema

DEXPI and PostProc coexist in the same tables with identical columns — the
interoperability promise made concrete.

In [14]:
spark.table("silver.silver_segments").groupBy("source_format").count().show()

+-------------+-----+
|source_format|count|
+-------------+-----+
|     POSTPROC|  807|
|        DEXPI|  713|
+-------------+-----+



### 3f. Real-data finding — `seg_tag` is not unique

Distinct `segment_id`s can compose to the **same** business `seg_tag`. So the
composed tag cannot stand alone as the CDC segment anchor — it needs a
disambiguator, and the quality gate owes an *anchor-collision* flag. (This is why
we recorded it in the spec's §3.5.)

In [15]:
(spark.table("silver.silver_segments")
   .groupBy("seg_tag").count().filter("count > 1")
   .orderBy(F.desc("count")).show(10, False))

+------------------------+-----+
|seg_tag                 |count|
+------------------------+-----+
|NULL                    |137  |
|1"-LS-36209-1S1A-1      |48   |
|1"-SV-36209-1B6AS-N     |47   |
|Conn to process/supply- |37   |
|44"-AG-36209-1C6AS-S    |27   |
|2"-LS-36209-1S1A-1      |25   |
|36"-PG-1417205-D24P1HD-H|24   |
|1.5"-LS-36209-1S1A-1    |23   |
|36"-PG-1418103-D341HD-H |22   |
|3"-SV-36209-1B6AS-N     |16   |
+------------------------+-----+
only showing top 10 rows



## 4. Recap

**Built (Phase-1):** Bronze (raw, immutable, dedup, format-tagged) → Silver (parse +
reconstruction, four typed tables), validated on real Project A **and** Project B.

**Concepts shown:** store-as-is + content hash; format detection; the reconstruction
recovering inline valves; the oracle firewall; the `flow_sense` enum and `derived`
provenance; format parity; and the `seg_tag` anchor-collision.

**Runtime lessons baked in:** force the venv's Spark 3.5.1 (Cell 1); pin the metastore
+ warehouse; run in-session; Bronze can be path-based to sidestep the catalog entirely.

**Next:** Stage C (OPC cross-document assembly), Stage D (Great-Expectations
punch-list → `silver_quality`, the engineer-facing data-quality list), Stage E
(object-grain CDC), then the Gold layer (bi-temporal + RDF/IDO).